# 03 — Optimization Walkthrough & Budget Sensitivity

This notebook walks through the 0-1 knapsack ILP formulation end-to-end,
solves it with OR-Tools CP-SAT, and runs a **budget sensitivity sweep** to
show the diminishing-returns curve a turnaround planner actually cares
about: *"what do we get for the next $500K?"*

## Formulation

Decision variable: `x_i ∈ {0, 1}` — include work order *i* in the turnaround.

**Maximize**  Σ value_i · x_i  (risk-adjusted net value)

**Subject to:**
- Σ cost_i · x_i ≤ Budget
- Σ mech_i · x_i ≤ MechHourCap   (and similarly for elec / inst / civil)
- x_i = 1  for every mandatory task
- x_j ≤ x_i  whenever task *j* requires predecessor task *i*


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.graph_objects as go
from copy import deepcopy

from src.utils.config import TA_CFG, SOLVER_CFG
from src.etl.extract import load_work_orders, load_failure_history
from src.etl.transform import run_transforms
from src.modeling.weibull import run_weibull_analysis
from src.modeling.risk import compute_risk_scores
from src.optimization.solver import TurnaroundSolver

raw_wos, raw_failures = load_work_orders(), load_failure_history()
clean_wos, clean_failures = run_transforms(raw_wos, raw_failures)
wos_reliability = run_weibull_analysis(clean_wos, clean_failures)
wos_scored      = compute_risk_scores(wos_reliability)

print(f"{len(wos_scored)} work orders ready for optimization")
print(f"Total backlog cost: ${wos_scored.estimated_cost_usd.sum():,.0f}")
print(f"Default budget:     ${TA_CFG.total_budget:,.0f}  "
      f"({TA_CFG.total_budget / wos_scored.estimated_cost_usd.sum():.1%} of backlog)")

2026-06-24 01:37:07  INFO      [etl.extract]  Extracting work orders from /home/claude/turnaround-optimizer/data/raw/work_orders.csv


2026-06-24 01:37:07  INFO      [etl.extract]    → 550 rows | 25 columns


2026-06-24 01:37:07  INFO      [helpers]  ⏱  load_work_orders completed in 0.012 s


2026-06-24 01:37:07  INFO      [etl.extract]  Extracting failure history from /home/claude/turnaround-optimizer/data/raw/failure_history.csv


2026-06-24 01:37:07  INFO      [etl.extract]    → 2467 failure events


...

2026-06-24 01:37:08  INFO      [helpers]  ⏱  run_weibull_analysis completed in 0.421 s


2026-06-24 01:37:08  INFO      [modeling.risk]  Risk scoring done | CRITICAL=33  HIGH=443  MEDIUM=66  LOW=8


2026-06-24 01:37:08  INFO      [modeling.risk]    Deferred risk portfolio: $12,389,715 | Mean P(fail)=0.617


2026-06-24 01:37:08  INFO      [helpers]  ⏱  compute_risk_scores completed in 0.042 s


550 work orders ready for optimization
Total backlog cost: $30,527,016
Default budget:     $5,000,000  (16.4% of backlog)


## 1. Solve at the Default Budget

In [2]:
result = TurnaroundSolver(wos_scored, config=TA_CFG).solve()

print(f"\nStatus:          {result.summary['solver_status']}")
print(f"Tasks selected:  {result.summary['tasks_selected']} / {result.summary['tasks_total']}")
print(f"Budget used:     ${result.summary['budget_used_usd']:,.0f} "
      f"({result.summary['budget_utilisation']:.1%} of cap)")
print(f"Net value:       ${result.summary['total_net_value_usd']:,.0f}")
print(f"ROI:             {result.summary['roi_ratio']:.2f}×")
print(f"Solve time:      {result.summary['solve_time_s']:.2f}s")

2026-06-24 01:37:08  INFO      [optimization.solver]  ============================================================


2026-06-24 01:37:08  INFO      [optimization.solver]  TURNAROUND ILP SOLVER  — OR-Tools CP-SAT


2026-06-24 01:37:08  INFO      [optimization.solver]    Tasks: 550 | Budget: $   5,000,000 | Timeout: 120.0s


2026-06-24 01:37:08  INFO      [optimization.solver]  ============================================================


2026-06-24 01:37:08  INFO      [optimization.model]  ILP model built | n=550 tasks | 550 variables | budget=$5,000,000


...

2026-06-24 01:37:08  INFO      [optimization.solver]    Inst hours      : 1,554 h  (25.9 % of capacity)


2026-06-24 01:37:08  INFO      [optimization.solver]    Net value       : $   4,710,074  (ROI 0.9×)


2026-06-24 01:37:08  INFO      [optimization.solver]    Risk score Δ    : 2742 units


2026-06-24 01:37:08  INFO      [optimization.solver]  ────────────────────────────────────────────────────────────



Status:          OPTIMAL
Tasks selected:  199 / 550
Budget used:     $4,999,232 (100.0% of cap)
Net value:       $4,710,074
ROI:             0.94×
Solve time:      0.08s


In [3]:
result.selected_schedule.sort_values("net_value_usd", ascending=False)[
    ["wo_id", "description", "priority", "estimated_cost_usd", "failure_prob",
     "risk_level", "net_value_usd"]
].head(15)

,wo_id,description,priority,estimated_cost_usd,failure_prob,risk_level,net_value_usd
382,WO-00383,Cleaning – Reciprocating Compressor CMP-0001,Low,1651.36,1.000000,CRITICAL,347473.64
387,WO-00388,Cleaning – Reciprocating Compressor CMP-0005,Low,2766.69,1.000000,CRITICAL,346358.31
500,WO-00501,Inspection – Reciprocating Compressor CMP-0017,Medium,6550.87,1.000000,CRITICAL,342574.13
448,WO-00449,Inspection – Reciprocating Compressor CMP-0007,Medium,3247.37,0.972771,CRITICAL,336371.15
141,WO-00142,Cleaning – Reciprocating Compressor CMP-0010,High,8371.15,0.880817,CRITICAL,299144.18
531,WO-00532,Calibration – Reciprocating Compressor CMP-0008,High,1126.86,0.807521,CRITICAL,280799.08
427,WO-00428,Inspection – Distillation Tower TWR-0007,Low,3326.04,0.369472,CRITICAL,268236.13
437,WO-00438,Cleaning – Distillation Tower TWR-0007,Critical,12497.54,0.369472,CRITICAL,259064.63
63,WO-00064,Replacement – Reciprocating Compressor CMP-0001,High,94600.93,1.000000,CRITICAL,254524.07
535,WO-00536,Replacement – Reciprocating Compressor CMP-0007,Medium,87251.78,0.972771,CRITICAL,252366.74


## 2. What Got Deferred? (the highest-value misses)

In [4]:
result.deferred_schedule.sort_values("net_value_usd", ascending=False)[
    ["wo_id", "description", "priority", "estimated_cost_usd", "failure_prob",
     "risk_level", "net_value_usd"]
].head(15)

,wo_id,description,priority,estimated_cost_usd,failure_prob,risk_level,net_value_usd
69,WO-00070,Cleaning – Pressure Vessel VSL-0012,High,24206.73,0.363962,CRITICAL,54954.90
392,WO-00393,Testing – Heat Exchanger HX-0034,Low,14454.01,0.266196,HIGH,16690.89
110,WO-00111,Replacement – Pressure Vessel VSL-0028,High,44519.28,0.270722,HIGH,14362.82
459,WO-00460,Calibration – Centrifugal Pump PMP-0010,Critical,2205.74,0.615983,HIGH,11099.49
357,WO-00358,Inspection – Centrifugal Pump PMP-0059,Medium,3613.66,0.634297,HIGH,10087.15
511,WO-00512,Calibration – Piping/Pipeline Segment PPL-0025,Medium,2917.45,0.184089,HIGH,5951.96
353,WO-00354,Cleaning – Centrifugal Pump PMP-0044,High,12152.47,0.715828,HIGH,3309.42
238,WO-00239,Calibration – Control Valve VLV-0106,Medium,1002.14,0.642365,HIGH,2444.95
332,WO-00333,Inspection – Centrifugal Pump PMP-0051,High,7332.46,0.433093,HIGH,2022.34
506,WO-00507,Inspection – Electrical Equipment ELEC-0061,High,5039.02,0.772355,HIGH,1912.17


## 3. Budget Sensitivity Sweep

The single most useful chart for a turnaround planning meeting: how does
total risk-adjusted value scale as budget increases? This identifies the
point of diminishing returns — additional dollars stop buying meaningful
risk reduction once the highest-value tasks are already funded.


In [5]:
budget_levels = [1_000_000, 2_000_000, 3_000_000, 4_000_000,
                 5_000_000, 6_500_000, 8_000_000, 10_000_000]

sweep_rows = []
for b in budget_levels:
    cfg = deepcopy(TA_CFG)
    cfg.total_budget = b
    try:
        r = TurnaroundSolver(wos_scored, config=cfg).solve()
        sweep_rows.append({
            "budget": b,
            "tasks_selected": r.summary["tasks_selected"],
            "net_value": r.summary["total_net_value_usd"],
            "budget_used": r.summary["budget_used_usd"],
            "mech_utilisation": r.summary["mech_utilisation"],
        })
    except RuntimeError:
        sweep_rows.append({"budget": b, "tasks_selected": None, "net_value": None,
                           "budget_used": None, "mech_utilisation": None})

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

2026-06-24 01:37:08  INFO      [optimization.solver]  ============================================================


2026-06-24 01:37:08  INFO      [optimization.solver]  TURNAROUND ILP SOLVER  — OR-Tools CP-SAT


2026-06-24 01:37:08  INFO      [optimization.solver]    Tasks: 550 | Budget: $   1,000,000 | Timeout: 120.0s


2026-06-24 01:37:08  INFO      [optimization.solver]  ============================================================


2026-06-24 01:37:08  INFO      [optimization.model]  ILP model built | n=550 tasks | 550 variables | budget=$1,000,000


...

2026-06-24 01:37:09  INFO      [optimization.solver]    Inst hours      : 1,898 h  (31.6 % of capacity)


2026-06-24 01:37:09  INFO      [optimization.solver]    Net value       : $   4,058,907  (ROI 0.7×)


2026-06-24 01:37:09  INFO      [optimization.solver]    Risk score Δ    : 3134 units


2026-06-24 01:37:09  INFO      [optimization.solver]  ────────────────────────────────────────────────────────────


,budget,tasks_selected,net_value,budget_used,mech_utilisation
0,1000000,NaN,NaN,NaN,NaN
1,2000000,NaN,NaN,NaN,NaN
2,3000000,NaN,NaN,NaN,NaN
3,4000000,144.0,4033297.22,3999909.01,0.1554
4,5000000,199.0,4710073.69,4999232.39,0.2184
5,6500000,233.0,4058907.12,5968449.77,0.2755
6,8000000,233.0,4058907.12,5968449.77,0.2755
7,10000000,233.0,4058907.12,5968449.77,0.2755


In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sweep_df.budget, y=sweep_df.net_value,
    mode="lines+markers", name="Net Risk-Adjusted Value",
    line=dict(color="#22c55e", width=3), marker=dict(size=9),
))
fig.add_trace(go.Scatter(
    x=[TA_CFG.total_budget], y=[result.summary["total_net_value_usd"]],
    mode="markers", name="Current Default Budget",
    marker=dict(size=16, color="#ef4444", symbol="star"),
))
fig.update_layout(
    title="Budget Sensitivity — Net Value vs. Available Budget",
    xaxis_title="Turnaround Budget (USD)", yaxis_title="Net Risk-Adjusted Value (USD)",
    template="plotly_white", height=460,
)
fig.show()

print("Marginal value of each additional budget increment:")
for i in range(1, len(sweep_df)):
    d_budget = sweep_df.budget[i] - sweep_df.budget[i-1]
    d_value  = sweep_df.net_value[i] - sweep_df.net_value[i-1]
    if d_budget > 0 and d_value is not None:
        print(f"  ${sweep_df.budget[i-1]:>10,.0f} → ${sweep_df.budget[i]:>10,.0f}:  "
              f"+${d_value:>10,.0f} value per +${d_budget:,.0f} spent  "
              f"(marginal ROI {d_value/d_budget:.2f}×)")

Marginal value of each additional budget increment:
  $ 1,000,000 → $ 2,000,000:  +$       nan value per +$1,000,000 spent  (marginal ROI nan×)
  $ 2,000,000 → $ 3,000,000:  +$       nan value per +$1,000,000 spent  (marginal ROI nan×)
  $ 3,000,000 → $ 4,000,000:  +$       nan value per +$1,000,000 spent  (marginal ROI nan×)
  $ 4,000,000 → $ 5,000,000:  +$   676,776 value per +$1,000,000 spent  (marginal ROI 0.68×)
  $ 5,000,000 → $ 6,500,000:  +$  -651,167 value per +$1,500,000 spent  (marginal ROI -0.43×)
  $ 6,500,000 → $ 8,000,000:  +$         0 value per +$1,500,000 spent  (marginal ROI 0.00×)
  $ 8,000,000 → $10,000,000:  +$         0 value per +$2,000,000 spent  (marginal ROI 0.00×)


In [7]:
fig = go.Figure()
fig.add_trace(go.Bar(x=sweep_df.budget, y=sweep_df.tasks_selected, marker_color="#3b82f6"))
fig.update_layout(
    title="Tasks Selected vs. Budget Level",
    xaxis_title="Turnaround Budget (USD)", yaxis_title="Number of Work Orders Selected",
    template="plotly_white", height=400,
)
fig.show()

## Conclusions

1. **Diminishing marginal ROI** — each additional budget increment buys
   progressively less risk-adjusted value as the solver works through its
   priority-ranked list. This is the expected, economically rational
   behavior of a knapsack solution and is a useful talking point when
   justifying (or declining) a budget increase request.
2. **Mandatory tasks are budget-inelastic** — the floor spend required by
   statutory/safety-mandatory work orders doesn't change with budget; only
   the *discretionary* pool above that floor responds to budget changes.
3. **Craft-hour capacity, not just budget, can bind** — check the
   `mech_utilisation` column in the sweep above; if it's pinned near 100%
   even as budget increases, the real constraint is crew availability, not
   money, and the conversation with leadership should shift accordingly.
